# Webdataset reader example in rocAL
This example demonstrates how to set up a simple webdataset reader pipeline. We load and decode image data stored in tar files using rocAL. The input data used for this example are images and labels saved as .tar file)

<font size="12"> Common Code </font>

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
import random
import numpy as np
from amd.rocal.plugin.generic import ROCALClassificationIterator
from amd.rocal.pipeline import Pipeline
import amd.rocal.fn as fn
import amd.rocal.types as types
import matplotlib.pyplot as plt
import os
%matplotlib inline

<font size= "12" >Configuring rocAL pipeline </font>

<div class="alert alert-block alert-warning">
<b>Note:</b> Set the ROCAL_DATA_PATH environment variable before running the notebook.
</div>

In [ ]:
# Check if ROCAL_DATA_PATH is set
rocal_data_path = os.environ.get('ROCAL_DATA_PATH')
if rocal_data_path is None:
    raise EnvironmentError("ROCAL_DATA_PATH environment variable is not set. Please set it to the correct path.")
if rocal_data_path is None:
    print("The environment variable ROCAL_DATA_PATH is not set.")
else:
    print(f"ROCAL_DATA_PATH IS SET TO: {rocal_data_path}")
wds_data_path = f"{rocal_data_path}/rocal_data/web_dataset/tar_file/"

Configure the pipeline parameters as required by the user.

In [ ]:
rocal_cpu = True
batch_size =  1
num_threads = 4
device_id = 0
random_seed = random.SystemRandom().randint(0, 2**32 - 1)
local_rank = 0
world_size = 1
color_format=types.RGB

In [ ]:
webdataset_pipeline = Pipeline(batch_size=batch_size, num_threads=num_threads, device_id=device_id, seed=random_seed, rocal_cpu=rocal_cpu)

<font size="12">Webdataset pipeline </font>

Here the webdataset reader is used followed by the webdataset decoder. In this pipeline, cascaded augmentations are added on the decoded images.<br>crop mirror normalize augmentation outputs are returned using set_outputs

In [ ]:
with webdataset_pipeline:
        img_raw = fn.readers.webdataset(path=wds_data_path, ext=[{'JPEG', 'cls'}], missing_components_behavior=types.MISSING_COMPONENT_SKIP)
        img = fn.decoders.image(img_raw, file_root=wds_data_path, output_type=color_format,max_decoded_width=416, max_decoded_height=416)
        resize_outputs = fn.resize(img, resize_width=300, resize_height=300)
        output = fn.crop_mirror_normalize(resize_outputs,
                                          output_layout=types.NHWC,
                                          output_dtype=types.UINT8,
                                          crop=(224, 224),
                                          mean=[0.0, 0.0, 0.0],
                                          std=[1.0, 1.0, 1.0])
        webdataset_pipeline.set_outputs(output)

In [ ]:
webdataset_pipeline.build()
data_loader = ROCALClassificationIterator(webdataset_pipeline)

<font size ="12">Visualizing  outputs</font>

The output of augmented images are displayed using imshow()

In [ ]:
cnt = 0
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(15,15))
row = 0
col = 0
for i, it in enumerate(data_loader, 0):
    for img in it[0]:
        img[0] = img[0].astype(np.uint8)
        axes[row, col].imshow(img[0])
        cnt += 1
        row += 1
        if(row == 2):
            row = 0
            col += 1
        if(col == 4):
            col = 0
data_loader.reset()